# Notebook 03 — HF_v6 Domain-Filtered Validation

Reproduces the per-domain Harmony Field correlation analysis.

**Key question:** Does restricting HF computation to the planetary significators
of a life domain (house) improve correlation with biographical events in that domain?

**Inputs:** `data/results/domain_correlation_table.csv`  
**Outputs:** `figures/domain_correlation_heatmap.png`, `figures/cohens_d_by_domain.png`

**Note on N discrepancies:** The domain_correlation_table uses N values from the
authoritative `domain_correlation_results.json` which was computed from the
private `biographical_events_v2/` dataset with per-event house labels.
The `config/event_house_map.json` provides a heuristic approximation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

REPO = Path('.')

table = pd.read_csv(REPO / 'data/results/domain_correlation_table.csv')
print('Domain correlation table:')
print(table.to_string(index=False))

In [ ]:
# Filter to houses with n >= 10
t = table[table['n_total'] >= 10].copy()
print(f'Houses with n>=10: {t["domain_name"].tolist()}')

# Improvement in Pearson r: domain vs global
t['delta_r'] = t['pearson_r_domain'] - t['pearson_r_global']
print()
print('Delta r (domain - global):')
print(t[['domain_name','n_total','pearson_r_global','pearson_r_domain','delta_r']].to_string(index=False))

In [ ]:
# Cohen's d comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

t_sorted = t.sort_values('cohens_d_domain')

axes[0].barh(t_sorted['domain_name'], t_sorted['cohens_d_global'].fillna(0),
             color='#C9A84C', alpha=0.7)
axes[0].axvline(0, color='white', lw=1)
axes[0].set_title("Cohen's d GLOBAL")
axes[0].set_xlabel("Cohen's d")

axes[1].barh(t_sorted['domain_name'], t_sorted['cohens_d_domain'].fillna(0),
             color='#C9A84C', alpha=0.7)
axes[1].axvline(0, color='white', lw=1)
axes[1].set_title("Cohen's d DOMAIN-FILTERED")
axes[1].set_xlabel("Cohen's d")

plt.suptitle('Effect Size: Global vs Domain-Filtered HF')
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap
hm = t.set_index('domain_name')[['pearson_r_global','pearson_r_domain',
                                  'cohens_d_global','cohens_d_domain']].astype(float)
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(hm, cmap='RdYlGn', center=0, annot=True, fmt='.2f', ax=ax,
            linewidths=0.5)
ax.set_title('HF Domain-Filtered Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# Summary conclusions
print('Summary:')
print(f'  Total houses analysed (n>=10): {len(t)}')
print(f'  Houses where domain r > global r: {(t["pearson_r_domain"] > t["pearson_r_global"]).sum()}')
print(f'  Houses where domain d > global d: {(t["cohens_d_domain"] > t["cohens_d_global"]).sum()}')

best = t.loc[t['cohens_d_domain'].idxmax() if t['cohens_d_domain'].notna().any() else t['pearson_r_domain'].idxmax()]
print(f'  Best domain by d_domain: {best["domain_name"]} (n={int(best["n_total"])})')